In [3]:
import sys
!{sys.executable} -m pip install wikirate4py pandas openpyxl


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: /Users/pablopadres/.pyenv/versions/3.11.8/bin/python -m pip install --upgrade pip


In [4]:
import wikirate4py
import pandas as pd
import numpy as np
import time
from pprint import pprint

api = wikirate4py.API("cMSYGt6SdESjs8h00EUIkwtt")

print("Imports loaded.")

Imports loaded.


In [5]:
brand_ids = [
    5505, 61104, 9410, 61050, 5800, 56013, 60982, 862441,
    60914, 116106, 116498, 2265395, 49386, 5449043,
    5448716, 5451022
]

brand_id_to_name = {
    5505: "GAP",
    61104: "VF Corp",
    9410: "Patagonia",
    61050: "PVH",
    5800: "Nike Inc.",
    56013: "New Balance",
    60982: "Hanesbrands",
    862441: "American Eagle",
    60914: "Abercrombie & Fitch",
    116106: "Fruit of the Loom",
    116498: "Under Armour",
    2265395: "Ralph Lauren",
    49386: "Tapestry",
    5449043: "Kontoor Brands",
    5448716: "Zegna",
    5451022: "Victoria’s Secret"
}

print("Total brands:", len(brand_ids))
print(brand_id_to_name)

Total brands: 16
{5505: 'GAP', 61104: 'VF Corp', 9410: 'Patagonia', 61050: 'PVH', 5800: 'Nike Inc.', 56013: 'New Balance', 60982: 'Hanesbrands', 862441: 'American Eagle', 60914: 'Abercrombie & Fitch', 116106: 'Fruit of the Loom', 116498: 'Under Armour', 2265395: 'Ralph Lauren', 49386: 'Tapestry', 5449043: 'Kontoor Brands', 5448716: 'Zegna', 5451022: 'Victoria’s Secret'}


In [6]:
def safe_json(item):
    try:
        return item.json()
    except Exception:
        try:
            return item.__dict__
        except Exception:
            return {"raw_item": str(item)}


def pull_relationships_safe(brand_id, max_rows=2000):
    rows = []

    cursor = wikirate4py.Cursor(
        api.get_relationships,
        metric_designer="Commons",
        metric_name="Supplied By",
        subject_company_id=brand_id,
        status="exists",
        per_page=50
    )

    while cursor.has_next():
        try:
            results = cursor.next()
        except Exception as e:
            if "429" in str(e):
                print("Rate limited — sleeping 60s")
                time.sleep(2)
                continue
            else:
                raise e

        for item in results:
            rows.append(safe_json(item))

            if len(rows) >= max_rows:
                return pd.json_normalize(rows)

        time.sleep(2)

    return pd.json_normalize(rows) if rows else pd.DataFrame()


print("Helper ready.")

Helper ready.


In [5]:
relationship_batches = []

for i, brand_id in enumerate(brand_ids):
    brand_name_manual = brand_id_to_name.get(brand_id, "N/A")
    
    print(f"\nPulling {i+1}/{len(brand_ids)}: {brand_name_manual} | ID {brand_id}")
    
    try:
        temp_df = pull_relationships_safe(
            brand_id=brand_id,
            max_rows=2000
        )
        
        print(f"Rows pulled: {len(temp_df)}")
        
        if temp_df.empty:
            temp_df = pd.DataFrame([{
                "manual_brand_name": brand_name_manual,
                "manual_brand_id": brand_id,
                "missing_note": "No supplier relationships returned"
            }])
        else:
            temp_df["manual_brand_name"] = brand_name_manual
            temp_df["manual_brand_id"] = brand_id
        
        relationship_batches.append(temp_df)
        
        time.sleep(5)
        
    except Exception as e:
        print(f"ERROR for {brand_name_manual}: {e}")
        relationship_batches.append(pd.DataFrame([{
            "manual_brand_name": brand_name_manual,
            "manual_brand_id": brand_id,
            "missing_note": str(e)
        }]))

relationships_raw = pd.concat(relationship_batches, ignore_index=True)

print("\nDone.")
print("Total rows:", len(relationships_raw))
print("Columns:")
print(relationships_raw.columns.tolist())

display(relationships_raw.head().T)


Pulling 1/16: GAP | ID 5505
Rows pulled: 1862

Pulling 2/16: VF Corp | ID 61104
Rows pulled: 2000

Pulling 3/16: Patagonia | ID 9410
Rows pulled: 305

Pulling 4/16: PVH | ID 61050
Rows pulled: 2000

Pulling 5/16: Nike Inc. | ID 5800
Rows pulled: 2000

Pulling 6/16: New Balance | ID 56013
Rows pulled: 996

Pulling 7/16: Hanesbrands | ID 60982
Rows pulled: 379

Pulling 8/16: American Eagle | ID 862441
Rows pulled: 1

Pulling 9/16: Abercrombie & Fitch | ID 60914
Rows pulled: 286

Pulling 10/16: Fruit of the Loom | ID 116106
Rows pulled: 595

Pulling 11/16: Under Armour | ID 116498
Rows pulled: 323

Pulling 12/16: Ralph Lauren | ID 2265395
Rows pulled: 384

Pulling 13/16: Tapestry | ID 49386
Rows pulled: 47

Pulling 14/16: Kontoor Brands | ID 5449043
Rows pulled: 876

Pulling 15/16: Zegna | ID 5448716
Rows pulled: 111

Pulling 16/16: Victoria’s Secret | ID 5451022
Rows pulled: 106

Done.
Total rows: 12271
Columns:
['id', 'metric', 'metric_id', 'value', 'year', 'comments', 'sources', 'subj

,0,1,2,3,4
id,6265767,6265773,6265776,6265779,6265782
metric,Commons+Supplied By,Commons+Supplied By,Commons+Supplied By,Commons+Supplied By,Commons+Supplied By
metric_id,2929009,2929009,2929009,2929009,2929009
value,Tier 1 Supplier,Tier 1 Supplier,Tier 1 Supplier,Tier 1 Supplier,Tier 1 Supplier
year,2020,2020,2020,2020,2020
comments,,,,,
sources,[],[],[],[],[]
subject_company_name,Gap inc.,Gap inc.,Gap inc.,Gap inc.,Gap inc.
object_company_name,Alif Embroidery Village Ltd.,Ananta Denim Technology Ltd,Ananta Garments Ltd,Colossus Apparel Limited,Colossus Apparel Limited unit 2
subject_company_id,5505,5505,5505,5505,5505


In [6]:
supplier_relationships = relationships_raw[
    [
        "manual_brand_name",
        "manual_brand_id",
        "subject_company_name",
        "subject_company_id",
        "object_company_name",
        "object_company_id",
        "value",
        "year",
        "sources",
        "url"
    ]
].copy()

supplier_relationships = supplier_relationships.rename(columns={
    "manual_brand_name": "brand_name_manual",
    "manual_brand_id": "brand_id_manual",
    "subject_company_name": "brand_name_wikirate",
    "subject_company_id": "brand_id_wikirate",
    "object_company_name": "facility_name",
    "object_company_id": "facility_id",
    "value": "supplier_relationship_type",
    "url": "relationship_url"
})

supplier_relationships = supplier_relationships.replace({None: "N/A", np.nan: "N/A", "": "N/A"})

for col in supplier_relationships.columns:
    supplier_relationships[col] = supplier_relationships[col].apply(
        lambda x: str(x) if isinstance(x, (list, dict)) else x
    )

supplier_relationships = supplier_relationships.drop_duplicates()

unique_supplier_ids = (
    supplier_relationships["facility_id"]
    .replace("N/A", np.nan)
    .dropna()
    .astype(int)
    .drop_duplicates()
    .tolist()
)

print("Clean relationship rows:", len(supplier_relationships))
print("Unique supplier IDs:", len(unique_supplier_ids))
display(supplier_relationships.head())

Clean relationship rows: 12145
Unique supplier IDs: 6091


,brand_name_manual,brand_id_manual,brand_name_wikirate,brand_id_wikirate,facility_name,facility_id,supplier_relationship_type,year,sources,relationship_url
0,GAP,5505,Gap inc.,5505,Alif Embroidery Village Ltd.,6178119,Tier 1 Supplier,2020,[],https://wikirate.org/Commons+Supplied_By+Gap_i...
1,GAP,5505,Gap inc.,5505,Ananta Denim Technology Ltd,3219140,Tier 1 Supplier,2020,[],https://wikirate.org/Commons+Supplied_By+Gap_i...
2,GAP,5505,Gap inc.,5505,Ananta Garments Ltd,5070765,Tier 1 Supplier,2020,[],https://wikirate.org/Commons+Supplied_By+Gap_i...
3,GAP,5505,Gap inc.,5505,Colossus Apparel Limited,6175881,Tier 1 Supplier,2020,[],https://wikirate.org/Commons+Supplied_By+Gap_i...
4,GAP,5505,Gap inc.,5505,Colossus Apparel Limited unit 2,3651611,Tier 1 Supplier,2020,[],https://wikirate.org/Commons+Supplied_By+Gap_i...


In [7]:
test_supplier_id = unique_supplier_ids[0]

print("Testing supplier ID:", test_supplier_id)

try:
    test_company = api.get_company(test_supplier_id)
    test_data = test_company.json()
    print("Profile keys:")
    print(test_data.keys())
    pprint(test_data)
except Exception as e:
    print("get_company failed:", e)

Testing supplier ID: 6178119
Profile keys:
dict_keys(['id', 'name', 'headquarters', 'wikipedia_url', 'aliases', 'url', 'os_id', 'sec_cik', 'lei', 'isin', 'open_corporates', 'australian_business_number', 'uk_company_number'])
{'aliases': None,
 'australian_business_number': None,
 'headquarters': ['Bangladesh'],
 'id': 6178119,
 'isin': None,
 'lei': None,
 'name': 'Alif Embroidery Village Ltd.',
 'open_corporates': None,
 'os_id': 'BD2023041JKXK29',
 'sec_cik': None,
 'uk_company_number': None,
 'url': 'https://wikirate.org/Alif_Embroidery_Village_Ltd',
 'wikipedia_url': None}


In [8]:
supplier_profile_rows = []

test_supplier_ids = unique_supplier_ids[:25]

for i, supplier_id in enumerate(test_supplier_ids):
    print(f"Processing {i+1}/{len(test_supplier_ids)} | Supplier ID: {supplier_id}")

    try:
        company = api.get_company(supplier_id)
        data = company.json()

        headquarters = data.get("headquarters", "N/A")

        if isinstance(headquarters, list):
            headquarters_clean = ", ".join(headquarters) if headquarters else "N/A"
        else:
            headquarters_clean = headquarters if headquarters else "N/A"

        supplier_profile_rows.append({
            "facility_id": data.get("id", supplier_id),
            "facility_name_profile": data.get("name", "N/A"),
            "headquarters": headquarters_clean,
            "supplier_profile_url": data.get("url", "N/A"),
            "os_id": data.get("os_id", "N/A")
        })

        time.sleep(1.2)

    except Exception as e:
        print(f"Error for supplier {supplier_id}: {e}")
        supplier_profile_rows.append({
            "facility_id": supplier_id,
            "facility_name_profile": "N/A",
            "headquarters": "N/A",
            "supplier_profile_url": "N/A",
            "os_id": "N/A"
        })

supplier_profiles_test = pd.DataFrame(supplier_profile_rows)

display(supplier_profiles_test.head(25))
print("Rows:", len(supplier_profiles_test))

Processing 1/25 | Supplier ID: 6178119
Processing 2/25 | Supplier ID: 3219140
Processing 3/25 | Supplier ID: 5070765
Processing 4/25 | Supplier ID: 6175881
Processing 5/25 | Supplier ID: 3651611
Processing 6/25 | Supplier ID: 3457603
Processing 7/25 | Supplier ID: 3244499
Processing 8/25 | Supplier ID: 4712737
Processing 9/25 | Supplier ID: 3219175
Processing 10/25 | Supplier ID: 3451838
Processing 11/25 | Supplier ID: 5106602
Processing 12/25 | Supplier ID: 5022601
Processing 13/25 | Supplier ID: 3222889
Processing 14/25 | Supplier ID: 4713704
Processing 15/25 | Supplier ID: 5220361
Processing 16/25 | Supplier ID: 3438850
Processing 17/25 | Supplier ID: 4273315
Processing 18/25 | Supplier ID: 3250244
Processing 19/25 | Supplier ID: 5073005
Processing 20/25 | Supplier ID: 3438932
Processing 21/25 | Supplier ID: 3634376
Processing 22/25 | Supplier ID: 3634362
Processing 23/25 | Supplier ID: 6176033
Processing 24/25 | Supplier ID: 3439017
Processing 25/25 | Supplier ID: 6162921


,facility_id,facility_name_profile,headquarters,supplier_profile_url,os_id
0,6178119,Alif Embroidery Village Ltd.,Bangladesh,https://wikirate.org/Alif_Embroidery_Village_Ltd,BD2023041JKXK29
1,3219140,Ananta Denim Technology Ltd,Bangladesh,https://wikirate.org/Ananta_Denim_Technology_Ltd,BD20190863HNZEP
2,5070765,Ananta Garments Ltd,Bangladesh,https://wikirate.org/Ananta_Garments_Ltd,BD2021341C364T6
3,6175881,Colossus Apparel Limited,Bangladesh,https://wikirate.org/Colossus_Apparel_Limited,BD2019091ZKVVVZ
4,3651611,Colossus Apparel Limited unit 2,Bangladesh,https://wikirate.org/Colossus_Apparel_Limited_...,BD2019091ZKVVVZ
5,3457603,Cortz Apparels Ltd.,Bangladesh,https://wikirate.org/Cortz_Apparels_Ltd,BD2021333FKGAS9
6,3244499,Creative Collection Ltd.,Bangladesh,https://wikirate.org/Creative_Collection_Ltd,BD2019133KEDYTZ
7,4712737,CROWN WEARS (PVT) LTD,Bangladesh,https://wikirate.org/CROWN_WEARS_PVT_LTD,BD2019083X6XXGA
8,3219175,Denimach Ltd,Bangladesh,https://wikirate.org/Denimach_Ltd,BD20190863XSXH7
9,3451838,Jeans 2000 Ltd,Bangladesh,https://wikirate.org/Jeans_2000_Ltd,BD20191085S9C5K


Rows: 25


In [9]:
supplier_profile_rows = []

for i, supplier_id in enumerate(unique_supplier_ids):
    if i % 100 == 0:
        print(f"Processing {i+1}/{len(unique_supplier_ids)}")

    try:
        company = api.get_company(supplier_id)
        data = company.json()

        headquarters = data.get("headquarters", "N/A")

        if isinstance(headquarters, list):
            headquarters_clean = ", ".join(headquarters) if headquarters else "N/A"
        else:
            headquarters_clean = headquarters if headquarters else "N/A"

        supplier_profile_rows.append({
            "facility_id": data.get("id", supplier_id),
            "facility_name_profile": data.get("name", "N/A"),
            "headquarters": headquarters_clean,
            "supplier_profile_url": data.get("url", "N/A"),
            "os_id": data.get("os_id", "N/A")
        })

    except Exception as e:
        print(f"Error for supplier {supplier_id}: {e}")
        supplier_profile_rows.append({
            "facility_id": supplier_id,
            "facility_name_profile": "N/A",
            "headquarters": "N/A",
            "supplier_profile_url": "N/A",
            "os_id": "N/A"
        })

    if (i + 1) % 500 == 0:
        checkpoint_df = pd.DataFrame(supplier_profile_rows)
        checkpoint_df.to_excel(f"supplier_profiles_checkpoint_{i+1}.xlsx", index=False)
        print(f"Checkpoint saved at {i+1} suppliers")

    time.sleep(1.1)

supplier_profiles = pd.DataFrame(supplier_profile_rows)

supplier_profiles.to_excel("supplier_profiles_full.xlsx", index=False)

print("Done.")
print("Supplier profile rows:", len(supplier_profiles))
display(supplier_profiles.head())

Processing 1/6091
Processing 101/6091
Processing 201/6091
Processing 301/6091
Processing 401/6091
Checkpoint saved at 500 suppliers
Processing 501/6091
Processing 601/6091
Processing 701/6091
Processing 801/6091
Processing 901/6091
Checkpoint saved at 1000 suppliers
Processing 1001/6091
Processing 1101/6091
Processing 1201/6091
Processing 1301/6091
Processing 1401/6091
Checkpoint saved at 1500 suppliers
Processing 1501/6091
Processing 1601/6091
Processing 1701/6091
Processing 1801/6091
Processing 1901/6091
Checkpoint saved at 2000 suppliers
Processing 2001/6091
Processing 2101/6091
Processing 2201/6091
Processing 2301/6091
Processing 2401/6091
Checkpoint saved at 2500 suppliers
Processing 2501/6091
Processing 2601/6091
Processing 2701/6091
Processing 2801/6091
Processing 2901/6091
Checkpoint saved at 3000 suppliers
Processing 3001/6091
Error for supplier 7287063: list index out of range
Processing 3101/6091
Processing 3201/6091
Processing 3301/6091
Processing 3401/6091
Checkpoint saved

,facility_id,facility_name_profile,headquarters,supplier_profile_url,os_id
0,6178119,Alif Embroidery Village Ltd.,Bangladesh,https://wikirate.org/Alif_Embroidery_Village_Ltd,BD2023041JKXK29
1,3219140,Ananta Denim Technology Ltd,Bangladesh,https://wikirate.org/Ananta_Denim_Technology_Ltd,BD20190863HNZEP
2,5070765,Ananta Garments Ltd,Bangladesh,https://wikirate.org/Ananta_Garments_Ltd,BD2021341C364T6
3,6175881,Colossus Apparel Limited,Bangladesh,https://wikirate.org/Colossus_Apparel_Limited,BD2019091ZKVVVZ
4,3651611,Colossus Apparel Limited unit 2,Bangladesh,https://wikirate.org/Colossus_Apparel_Limited_...,BD2019091ZKVVVZ


In [10]:
supplier_network = supplier_relationships.merge(
    supplier_profiles,
    on="facility_id",
    how="left"
)

supplier_network["headquarters"] = supplier_network["headquarters"].replace({
    None: "N/A",
    np.nan: "N/A",
    "": "N/A"
})

display(supplier_network.head())

print("Rows:", len(supplier_network))
print("Unique brands:", supplier_network["brand_name_manual"].nunique())
print("Unique facilities:", supplier_network["facility_id"].nunique())
print("Rows missing headquarters:", (supplier_network["headquarters"] == "N/A").sum())

,brand_name_manual,brand_id_manual,brand_name_wikirate,brand_id_wikirate,facility_name,facility_id,supplier_relationship_type,year,sources,relationship_url,facility_name_profile,headquarters,supplier_profile_url,os_id
0,GAP,5505,Gap inc.,5505,Alif Embroidery Village Ltd.,6178119,Tier 1 Supplier,2020,[],https://wikirate.org/Commons+Supplied_By+Gap_i...,Alif Embroidery Village Ltd.,Bangladesh,https://wikirate.org/Alif_Embroidery_Village_Ltd,BD2023041JKXK29
1,GAP,5505,Gap inc.,5505,Ananta Denim Technology Ltd,3219140,Tier 1 Supplier,2020,[],https://wikirate.org/Commons+Supplied_By+Gap_i...,Ananta Denim Technology Ltd,Bangladesh,https://wikirate.org/Ananta_Denim_Technology_Ltd,BD20190863HNZEP
2,GAP,5505,Gap inc.,5505,Ananta Garments Ltd,5070765,Tier 1 Supplier,2020,[],https://wikirate.org/Commons+Supplied_By+Gap_i...,Ananta Garments Ltd,Bangladesh,https://wikirate.org/Ananta_Garments_Ltd,BD2021341C364T6
3,GAP,5505,Gap inc.,5505,Colossus Apparel Limited,6175881,Tier 1 Supplier,2020,[],https://wikirate.org/Commons+Supplied_By+Gap_i...,Colossus Apparel Limited,Bangladesh,https://wikirate.org/Colossus_Apparel_Limited,BD2019091ZKVVVZ
4,GAP,5505,Gap inc.,5505,Colossus Apparel Limited unit 2,3651611,Tier 1 Supplier,2020,[],https://wikirate.org/Commons+Supplied_By+Gap_i...,Colossus Apparel Limited unit 2,Bangladesh,https://wikirate.org/Colossus_Apparel_Limited_...,BD2019091ZKVVVZ


Rows: 12145
Unique brands: 16
Unique facilities: 6091
Rows missing headquarters: 162


In [11]:
supplier_network_clean = supplier_network[
    [
        "brand_name_manual",
        "brand_id_manual",
        "brand_name_wikirate",
        "facility_name",
        "facility_id",
        "headquarters",
        "year",
        "supplier_relationship_type",
        "relationship_url",
        "supplier_profile_url",
        "os_id"
    ]
].copy()

supplier_network_clean = supplier_network_clean.rename(columns={
    "brand_name_manual": "brand",
    "brand_id_manual": "brand_id",
    "brand_name_wikirate": "brand_wikirate_name",
    "headquarters": "country"
})

supplier_network_clean = supplier_network_clean.replace({
    None: "N/A",
    np.nan: "N/A",
    "": "N/A"
})

display(supplier_network_clean.head())
print("Rows:", len(supplier_network_clean))
print("Missing country:", (supplier_network_clean["country"] == "N/A").sum())

,brand,brand_id,brand_wikirate_name,facility_name,facility_id,country,year,supplier_relationship_type,relationship_url,supplier_profile_url,os_id
0,GAP,5505,Gap inc.,Alif Embroidery Village Ltd.,6178119,Bangladesh,2020,Tier 1 Supplier,https://wikirate.org/Commons+Supplied_By+Gap_i...,https://wikirate.org/Alif_Embroidery_Village_Ltd,BD2023041JKXK29
1,GAP,5505,Gap inc.,Ananta Denim Technology Ltd,3219140,Bangladesh,2020,Tier 1 Supplier,https://wikirate.org/Commons+Supplied_By+Gap_i...,https://wikirate.org/Ananta_Denim_Technology_Ltd,BD20190863HNZEP
2,GAP,5505,Gap inc.,Ananta Garments Ltd,5070765,Bangladesh,2020,Tier 1 Supplier,https://wikirate.org/Commons+Supplied_By+Gap_i...,https://wikirate.org/Ananta_Garments_Ltd,BD2021341C364T6
3,GAP,5505,Gap inc.,Colossus Apparel Limited,6175881,Bangladesh,2020,Tier 1 Supplier,https://wikirate.org/Commons+Supplied_By+Gap_i...,https://wikirate.org/Colossus_Apparel_Limited,BD2019091ZKVVVZ
4,GAP,5505,Gap inc.,Colossus Apparel Limited unit 2,3651611,Bangladesh,2020,Tier 1 Supplier,https://wikirate.org/Commons+Supplied_By+Gap_i...,https://wikirate.org/Colossus_Apparel_Limited_...,BD2019091ZKVVVZ


Rows: 12145
Missing country: 162


In [12]:
supplier_network_clean.to_excel(
    "brand_supplier_country_network_final.xlsx",
    index=False
)

print("Export complete.")

Export complete.


In [ ]:
##now im gonna do the final dataset for the metric below##

In [7]:
import os

print(os.getcwd())
print(os.listdir())

/Users/pablopadres/Documents/School/MS2/thesis/PCA
['supplier checkpoints', '.DS_Store', 'Untitled.ipynb', 'HRRR_metric.ipynb', 'Labor_Compensation.xlsx', 'HRRR_final.ipynb', 'brand_supplier_country_network_final.xlsx', '.ipynb_checkpoints']


In [9]:
supplier_network_clean = pd.read_excel(
    "brand_supplier_country_network_final.xlsx"
)

display(supplier_network_clean.head())

,brand,brand_id,brand_wikirate_name,facility_name,facility_id,country,year,supplier_relationship_type,relationship_url,supplier_profile_url,os_id
0,GAP,5505,Gap inc.,Alif Embroidery Village Ltd.,6178119,Bangladesh,2020,Tier 1 Supplier,https://wikirate.org/Commons+Supplied_By+Gap_i...,https://wikirate.org/Alif_Embroidery_Village_Ltd,BD2023041JKXK29
1,GAP,5505,Gap inc.,Ananta Denim Technology Ltd,3219140,Bangladesh,2020,Tier 1 Supplier,https://wikirate.org/Commons+Supplied_By+Gap_i...,https://wikirate.org/Ananta_Denim_Technology_Ltd,BD20190863HNZEP
2,GAP,5505,Gap inc.,Ananta Garments Ltd,5070765,Bangladesh,2020,Tier 1 Supplier,https://wikirate.org/Commons+Supplied_By+Gap_i...,https://wikirate.org/Ananta_Garments_Ltd,BD2021341C364T6
3,GAP,5505,Gap inc.,Colossus Apparel Limited,6175881,Bangladesh,2020,Tier 1 Supplier,https://wikirate.org/Commons+Supplied_By+Gap_i...,https://wikirate.org/Colossus_Apparel_Limited,BD2019091ZKVVVZ
4,GAP,5505,Gap inc.,Colossus Apparel Limited unit 2,3651611,Bangladesh,2020,Tier 1 Supplier,https://wikirate.org/Commons+Supplied_By+Gap_i...,https://wikirate.org/Colossus_Apparel_Limited_...,BD2019091ZKVVVZ


In [27]:
wageindicator_df = pd.read_excel("Labor_Compensation.xlsx")

display(wageindicator_df.head())
print("Countries:", len(wageindicator_df))

,Country,Monthly Minimum Wage in USD$,Avg Monthly Living Wage in USD$
0,Albania,614.68,485.00
1,Argentina,257.50,405.75
2,Australia,2718.09,1636.00
3,Bangladesh,102.06,242.08
4,Belgium,2567.42,1218.50


Countries: 111


In [28]:
wageindicator_df = wageindicator_df.rename(columns={
    "Country": "country",
    "Monthly Minimum Wage in USD$": "minimum_wage",
    "Avg Monthly Living Wage in USD$": "living_wage"
})

wageindicator_df["country"] = (
    wageindicator_df["country"]
    .astype(str)
    .str.lower()
    .str.strip()
)

supplier_network_clean["country"] = (
    supplier_network_clean["country"]
    .astype(str)
    .str.lower()
    .str.strip()
)

In [29]:
wageindicator_df["minimum_wage"] = (
    wageindicator_df["minimum_wage"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

wageindicator_df["living_wage"] = (
    wageindicator_df["living_wage"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

wageindicator_df["minimum_wage"] = pd.to_numeric(
    wageindicator_df["minimum_wage"], errors="coerce"
)

wageindicator_df["living_wage"] = pd.to_numeric(
    wageindicator_df["living_wage"], errors="coerce"
)

display(wageindicator_df.head())

,country,minimum_wage,living_wage
0,albania,614.68,485.00
1,argentina,257.50,405.75
2,australia,2718.09,1636.00
3,bangladesh,102.06,242.08
4,belgium,2567.42,1218.50


In [30]:
wageindicator_df["wage_ratio"] = (
    wageindicator_df["minimum_wage"] / wageindicator_df["living_wage"]
)

wageindicator_df["country_risk"] = 1 - wageindicator_df["wage_ratio"]

wageindicator_df["country_risk"] = wageindicator_df["country_risk"].clip(0, 1)

wageindicator_df["country_labor_score"] = 100 * (1 - wageindicator_df["country_risk"])
wageindicator_df["country_labor_score"] = wageindicator_df["country_labor_score"].round(2)

display(wageindicator_df[
    ["country", "minimum_wage", "living_wage", "wage_ratio", "country_risk", "country_labor_score"]
].head())

,country,minimum_wage,living_wage,wage_ratio,country_risk,country_labor_score
0,albania,614.68,485.00,1.267381,0.000000,100.00
1,argentina,257.50,405.75,0.634627,0.365373,63.46
2,australia,2718.09,1636.00,1.661424,0.000000,100.00
3,bangladesh,102.06,242.08,0.421596,0.578404,42.16
4,belgium,2567.42,1218.50,2.107033,0.000000,100.00


In [31]:
supplier_network_clean["country"] = (
    supplier_network_clean["country"]
    .astype(str)
    .str.lower()
    .str.strip()
)

wageindicator_df["country"] = (
    wageindicator_df["country"]
    .astype(str)
    .str.lower()
    .str.strip()
)

In [32]:
supplier_network_clean["country"] = supplier_network_clean["country"].replace({
    "united states": "usa"
})

In [33]:
supplier_with_risk = supplier_network_clean.merge(
    wageindicator_df[
        ["country", "minimum_wage", "living_wage", "wage_ratio", "country_risk", "country_labor_score"]
    ],
    on="country",
    how="left"
)

display(supplier_with_risk.head())

print("Rows:", len(supplier_with_risk))
print("Missing country risk:", supplier_with_risk["country_risk"].isna().sum())

,brand,brand_id,brand_wikirate_name,facility_name,facility_id,country,year,supplier_relationship_type,relationship_url,supplier_profile_url,os_id,minimum_wage,living_wage,wage_ratio,country_risk,country_labor_score
0,GAP,5505,Gap inc.,Alif Embroidery Village Ltd.,6178119,bangladesh,2020,Tier 1 Supplier,https://wikirate.org/Commons+Supplied_By+Gap_i...,https://wikirate.org/Alif_Embroidery_Village_Ltd,BD2023041JKXK29,102.06,242.08,0.421596,0.578404,42.16
1,GAP,5505,Gap inc.,Ananta Denim Technology Ltd,3219140,bangladesh,2020,Tier 1 Supplier,https://wikirate.org/Commons+Supplied_By+Gap_i...,https://wikirate.org/Ananta_Denim_Technology_Ltd,BD20190863HNZEP,102.06,242.08,0.421596,0.578404,42.16
2,GAP,5505,Gap inc.,Ananta Garments Ltd,5070765,bangladesh,2020,Tier 1 Supplier,https://wikirate.org/Commons+Supplied_By+Gap_i...,https://wikirate.org/Ananta_Garments_Ltd,BD2021341C364T6,102.06,242.08,0.421596,0.578404,42.16
3,GAP,5505,Gap inc.,Colossus Apparel Limited,6175881,bangladesh,2020,Tier 1 Supplier,https://wikirate.org/Commons+Supplied_By+Gap_i...,https://wikirate.org/Colossus_Apparel_Limited,BD2019091ZKVVVZ,102.06,242.08,0.421596,0.578404,42.16
4,GAP,5505,Gap inc.,Colossus Apparel Limited unit 2,3651611,bangladesh,2020,Tier 1 Supplier,https://wikirate.org/Commons+Supplied_By+Gap_i...,https://wikirate.org/Colossus_Apparel_Limited_...,BD2019091ZKVVVZ,102.06,242.08,0.421596,0.578404,42.16


Rows: 12145
Missing country risk: 162


In [34]:
missing_countries = (
    supplier_with_risk[supplier_with_risk["country_risk"].isna()]
    ["country"]
    .value_counts()
)

display(missing_countries.head(30))
print("Unique missing countries:", missing_countries.shape[0])

country
nan    162
Name: count, dtype: int64

Unique missing countries: 1


In [16]:
country_fix_map = {
    "united states": "United States",
    "tamil nadu": "India",
    "korea, republic of": "South Korea",
    "quebec (canada)": "Canada",
    "sharjah (united arab emirates)": "United Arab Emirates",
    "ajman (united arab emirates)": "United Arab Emirates"
}

supplier_network_clean["country"] = (
    supplier_network_clean["country"]
    .replace(country_fix_map)
)

In [37]:
scored_suppliers_unique = (
    scored_suppliers
    .drop_duplicates(subset=["brand", "facility_id"])
)

brand_labor_score = (
    scored_suppliers_unique
    .groupby("brand")
    .agg(
        unique_suppliers=("facility_id", "nunique"),
        avg_country_risk=("country_risk", "mean"),
        avg_wage_ratio=("wage_ratio", "mean")
    )
    .reset_index()
)

brand_labor_score["labor_compensation_score"] = (
    100 * (1 - brand_labor_score["avg_country_risk"])
).round(2)

brand_labor_score = brand_labor_score.sort_values(
    "labor_compensation_score",
    ascending=False
)

brand_labor_score["data_flag"] = np.where(
    brand_labor_score["unique_suppliers"] < 30,
    "LOW DATA",
    "OK"
)

display(brand_labor_score)

,brand,unique_suppliers,avg_country_risk,avg_wage_ratio,labor_compensation_score,data_flag
15,Zegna,72,0.019124,1.023529,98.09,OK
10,Ralph Lauren,383,0.344099,0.713087,65.59,OK
8,PVH,1000,0.476633,0.563019,52.34,OK
13,VF Corp,1310,0.481065,0.563280,51.89,OK
7,Nike Inc.,847,0.484484,0.568952,51.55,OK
11,Tapestry,46,0.488881,0.539014,51.11,OK
5,Kontoor Brands,656,0.497036,0.528732,50.30,OK
12,Under Armour,195,0.500201,0.523884,49.98,OK
4,Hanesbrands,320,0.517829,0.509920,48.22,OK
2,Fruit of the Loom,451,0.527951,0.494852,47.20,OK
